[Open in Colab](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_09.ipynb) · **Before editing: File → Save a copy in Drive.**

[Download notebook](/downloads/Week_09.ipynb) · [Weekly ZIP](/downloads/Week_09.zip)

Open in Colab and save a copy in Drive. Run the setup cell to restore embedded support files. No ZIP, upload or network download is needed. The single notebook also works in local Jupyter.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/loem-ms/programming-for-information-systems-site/blob/main/content/downloads/Week_09.ipynb)

**Before editing:** choose **File → Save a copy in Drive**.

[Course website](https://loem-ms.github.io/programming-for-information-systems-site/) · [Setup and saving](https://loem-ms.github.io/programming-for-information-systems-site/content/setup/)

**Ready to run:** this notebook contains its support files. Connect to a normal Python runtime and run the setup cell below. No ZIP, upload, support-file download, installation or API key is needed. Opening this link alone does not save a personal copy. Rerun setup after a runtime restart. Existing support files are preserved. Read and run one cell at a time; complete marked student tasks yourself. Save your notebook and download any edited or created files separately before the temporary runtime ends.


# Week 09 · Functions, filters and assertions


(week-09-start)=
### Start Here

This notebook contains **the lesson, practice and project work for Week 09**. Use Python 3 with a fresh kernel and keep the supplied programs, data and diagrams beside it. [Python Basics](https://loem-ms.github.io/programming-for-information-systems-site/content/learning-materials/python-basics/) is an optional syntax lookup; you do not need to open another notebook to complete today's activities.

| Go to | What you do |
|---|---|
| [Session 1 · Python foundations](#week-09-session-1) | Read a short explanation and follow a small example |
| [Session 2 · Hands-on practice](#week-09-session-2) | Predict, run, modify and check the examples below |
| [Session 3 · Project Studio](#week-09-project-studio) | Build a tested approved feature with a peer finding and retest |
| [Finish and save](#week-09-finish) | Check your evidence, save and choose the next step |

**Activity rhythm:** Read → Predict → Run → Modify → Check → Next. Student-work cells remain intentionally unfinished. A successful run of the untouched starter does not complete the tasks.

**Later-course focus:** these applied topics extend the basic Python core completed in Weeks 1–4. Use the supplied infrastructure and concentrate on the current behavior, evidence and explanation.

### Prepare your workspace

Run the supplied setup cell after every runtime restart. It restores embedded programs, data and diagrams without downloading or uploading anything. Existing files are never overwritten. This helper is not a programming exercise. Read the later task instructions before running unfinished student work.


In [ ]:
# @title Prepare the course files (run after each runtime restart)
# Supplied programs, starter policy, CSV fixtures and diagrams are embedded here.
# This cell uses only Python's standard library. It makes no network request.
import base64 as _course_base64
import hashlib as _course_hashlib
import json as _course_json
import os as _course_os
from pathlib import Path as _CoursePath
import zlib as _course_zlib

_course_payload = '...'
_course_raw = _course_zlib.decompress(_course_base64.b64decode(_course_payload))
if _course_hashlib.sha256(_course_raw).hexdigest() != '630743ae8f1887cfc5c83623756bb3e8436b2187930cc3680abc47c9b0758a6e':
    raise ValueError("Embedded course files are damaged; get a fresh notebook.")
_course_files = _course_json.loads(_course_raw)
_course_key = 'week_09_630743ae8f18'
_course_roots = globals().setdefault("_course_workspace_roots", {})
if _course_key in _course_roots:
    _course_root = _CoursePath(_course_roots[_course_key])
elif all((_CoursePath.cwd() / _name).is_file() for _name in _course_files):
    # A complete extracted offline pack can be used in place.
    _course_root = _CoursePath.cwd()
else:
    _course_root = _CoursePath.cwd() / ".course_files" / _course_key
if _course_root.is_symlink() or any(_p.is_symlink() for _p in _course_root.parents):
    raise ValueError("Refusing a symbolic link in the course workspace")
_course_root.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _course_files.items():
    _relative = _CoursePath(_name)
    if _relative.is_absolute() or ".." in _relative.parts:
        raise ValueError("Unsafe embedded support path")
    _target = _course_root / _relative
    if _target.is_symlink() or any(_p.is_symlink() for _p in _target.parents):
        raise ValueError("Refusing a symbolic link in the course workspace")
    _target.parent.mkdir(parents=True, exist_ok=True)
    if _target.exists() and not _target.is_file():
        raise ValueError("Course support path is not a regular file: " + str(_target))
    if not _target.exists():
        with _target.open("xb") as _output:
            _output.write(_course_base64.b64decode(_encoded))
_course_workspace_roots[_course_key] = str(_course_root.resolve())
_course_os.chdir(_course_root)
print("Course files ready:", _CoursePath.cwd())
print("Existing files were kept. Rerunning setup never replaces your edits.")
print("Save your notebook and download created/edited files before the runtime ends.")


## This week: required, supplied and optional

Start with the coordination need, connect it to the new Python words and syntax, trace the worked example, make the guided change, predict the independent variation, and apply it to the Session 3 component.

- **Required evidence:** Implement one approved small change, run its acceptance cases and keep one peer finding plus retest.
- **Already supplied — use it; do not rebuild it:** The baseline application, starter scaffold, fixtures and review form are supplied.
- **Optional extension:** A web UI, broad redesign or extra feature is not required.

**Stop point:** once the required evidence is saved and you can explain it, stop. You may return to optional material later.



(week-09-session-1)=
## Session 1 · Python foundations

Start with the general Python concept and the self-contained example below. Predict, run and explain it before writing your own code in Session 2. A short optional Food Rescue note shows one application; no project knowledge is needed for this lesson.

### Predicates, result lists and independent tests

A program often selects records that satisfy a condition. For example, a task list might contain enabled, finished and published flags. “Which enabled tasks are unfinished?” asks a different question from “Which tasks are unpublished?” A finished task can remain unpublished. State the question precisely, then choose an example that distinguishes it from a similar question before writing code.

A predicate is an expression or function whose result is True or False. For valid Boolean fields, `row["enabled"] and not row["finished"]` selects enabled tasks that are unfinished. The published field is irrelevant to this particular question. A different requirement may need it. Specify the input fields and their types so the function has a clear contract. The examples here assume all required fields exist and hold valid Booleans; parsing uncertain external data is a separate task.

An ordinary loop can build the result. Initialize an empty list before the loop. Visit each record, evaluate the predicate, and append the requested ID only when the condition is true. Return the completed list after the loop. Returning inside the loop would stop processing early. Appending outside the condition would include unwanted records. An empty input should naturally produce an empty result.

A list comprehension can express the same selection more compactly, but it is optional. Learn to explain the ordinary loop first. Preserve input order unless the contract asks for sorting. Build a new list rather than removing items from the list you are visiting. Avoid changing the original records while creating a report: another part of the program may still need those values.

Tests compare the observed result with an independent expectation. Write the expected IDs directly from the requirement before calling your function. Computing the expected result with the same predicate as the implementation can repeat the same mistake on both sides. Include an enabled finished task that is unpublished, a disabled unfinished task, and an empty list. These cases distinguish plausible wrong conditions. A regression test protects a result that should stay unchanged after a code change. An assertion passing on one ordinary case is not evidence that every condition works.

Optional application: Food Rescue can use the same filtering pattern to select records for a coordinator's question. Its field names and meanings differ, but the programming method remains a precise predicate, a new result and independently chosen tests. The task-list example needs no knowledge of that system.

### Learn one operation at a time

Read and run these small examples in order before the combined example. Predict each output, point to the operation that produced it, and pause for clarification if it is unclear. They explain the existing lesson; they are not an extra homework list or extra assessed deliverables. The practice cells later remain yours to write.


### Small example 1 · Make one predicate from one record

A predicate returns a Boolean. Use a single dictionary before filtering a whole list. Here finished is false and enabled is true, so predict True. published is irrelevant to the stated question.


In [ ]:
def needs_work(row):
    return row["enabled"] and not row["finished"]

row = {"enabled": True, "finished": False, "published": False}
print(needs_work(row))
assert needs_work(row) is True


### Small example 2 · Build a new result instead of changing input

Start with plain numbers so the selection operation is visible without records. The loop visits every input but appends only matching values. Predict [2, 3], then the unchanged original list. The worked example next transfers this pattern to dictionaries.


In [ ]:
numbers = [1, 2, 3]
selected = []
for number in numbers:
    if number >= 2:
        selected.append(number)
print(selected)
print(numbers)
assert numbers == [1, 2, 3]


### Small example 3 · Keep an independent before snapshot

A second name such as alias still refers to the original object. deepcopy makes an independent copy, including dictionaries inside a list. Predict 9 for alias, 1 for before. This is test support, not a new feature to build.


In [ ]:
import copy
records = [{"points": 1}]
alias = records
before = copy.deepcopy(records)
records[0]["points"] = 9
print(alias[0]["points"], before[0]["points"])
assert before == [{"points": 1}]


### Worked example: predict, then trace

Read the complete example once. Write each expected result before running. Point to the statement responsible for each value.

In [ ]:
tasks = [
    {"task_id": "A", "enabled": True, "finished": False, "published": False},
    {"task_id": "B", "enabled": True, "finished": True, "published": False},
    {"task_id": "C", "enabled": False, "finished": False, "published": False},
]
selected = []
for row in tasks:
    wanted = row["enabled"] and not row["finished"]
    print(row["task_id"], wanted, "Unpublished:", not row["published"])
    if wanted:
        selected.append(row["task_id"])
print("Selected:", selected)
assert selected == ["A"]


The selection predicate is True, False, False for A, B, C, so selected is ["A"]. All three are unpublished, demonstrating why that other field cannot answer the stated question. B tests the finished condition and C tests enabled.

### Ready to write

Explain the input, the result and one important Python step without reading the output. Continue to [Session 2](#week-09-session-2). Apply the same general skill to each problem after reading its own stated rule.


(week-09-session-2)=
## Session 2 · Hands-on practice

(week-09-practice-week-09)=
**Five leveled coding problems:** P1 and P2 are easy problems with approach hints, not ready-made code. P3 is the required core implementation. P4 applies the skill and P5 is a harder challenge. Complete P1–P3 first; P4–P5 are optional.

Write a prediction, then write the program yourself. Keep the first result, explain a mismatch and retest. You may use the Session 1 syntax explanation. Beginners can work with a partner and request a hint; faster learners should attempt P4–P5 instead of waiting. Each learner saves their own code and explanation.

### Practice route

- **P1–P2 · Easy:** Use the approach hints. Write the code yourself in a blank cell.
- **P3 · Core build:** Implement a complete small program. Write expected results and assertions.
- **P4–P5 · Optional:** Apply the skill to a new case, then attempt a harder challenge if time permits.

### Supplied practice records · run before P1

P1, P2 and P3 use the same four records below. This cell supplies data only; write each solution in its blank work cell. T11 awaits collection. T12 is collected but not delivered. T13 is not reserved. T14 is already delivered. Each flag is a valid Boolean. These examples do not prescribe your final-project feature.


In [ ]:
records = [
    {"request_id": "T11", "reserved": True, "collected": False, "delivered": False},
    {"request_id": "T12", "reserved": True, "collected": True, "delivered": False},
    {"request_id": "T13", "reserved": False, "collected": False, "delivered": False},
    {"request_id": "T14", "reserved": True, "collected": True, "delivered": True},
]


### P1 · Collect IDs with a loop

**Required core · Level 1.**

Easy. From the supplied practice-record cell immediately above, write a loop that appends every request_id to a new list named ids. Predict the four IDs in order. Hint: initialize the result before the loop and append inside it. Assert the complete expected list. Do not change the records.


In [ ]:
# Write your code here. Predict before running.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P2 · Write the selection predicate

**Required core · Level 1.**

Easy. Write needs_collection(row) returning True only when reserved is True and collected is False. Fields in this fixture are valid Booleans. Predict True, False, False, False for the four records and write assertions. Hint: combine the two relevant fields; delivered is not part of this question.

In [ ]:
# Write your code here. Predict before running.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P3 · Build and test the report helper

**Required core · Level 2.**

Core build. Write awaiting_collection(records), returning matching IDs in input order. Use an ordinary loop, a condition, append and return. Independently write expectations for the full fixture (["T11"]), empty input ([]), and two pending records (["T11","T15"]). Add T15 as a new dict rather than modifying T12. For unchanged-input evidence, first import copy, then save before = copy.deepcopy(records). After calling your helper, assert records == before. A deep copy keeps an independent snapshot; before = records would only create another name for the same list. Explain why T12 catches not-delivered filtering and T13 catches a missing reservation condition.


In [ ]:
records = [
    {"request_id": "T11", "reserved": True, "collected": False, "delivered": False},
    {"request_id": "T12", "reserved": True, "collected": True, "delivered": False},
    {"request_id": "T13", "reserved": False, "collected": False, "delivered": False},
    {"request_id": "T14", "reserved": True, "collected": True, "delivered": True},
]

def awaiting_collection(records):
    # Build and return a new list. Leave source records unchanged.
    raise NotImplementedError

# Write your expectations and tests before running the function.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P4 · Write a reusable regression check

**Optional extension · Level 3.**

Optional for learners who finish the core. Optional application. Write check_report(select_function) that calls a supplied function on your fixture and on [], asserting ["T11"] and [] respectively. Run it against your correct helper. Then write a deliberately wrong helper that selects not delivered and show that your test raises AssertionError. Catch that error only around the deliberate wrong version and keep the passing correct version.

In [ ]:
records = [
    {"request_id": "T11", "reserved": True, "collected": False, "delivered": False},
    {"request_id": "T12", "reserved": True, "collected": True, "delivered": False},
    {"request_id": "T13", "reserved": False, "collected": False, "delivered": False},
    {"request_id": "T14", "reserved": True, "collected": True, "delivered": True},
]

# Reuse awaiting_collection. Write check_report and the deliberate wrong copy.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

### P5 · Write unittest cases

**Optional extension · Level 4.**

Optional for learners who finish the core. Optional challenge. Write a unittest.TestCase class with separate methods for empty input, the distinguishing T12 record, and unchanged source records. Use assertEqual and copy.deepcopy. Run with TextTestRunner against an explicitly loaded suite so notebook arguments do not interfere. Use the supplied unittest reference only for runner syntax. No new feature is required.

In [ ]:
import unittest
import copy
# Reuse records and awaiting_collection.
# Write your TestCase and explicit suite/runner below.


**My evidence**

- Expected result before running: …
- Observed result and test input: …
- My code explanation or repair: …

Optional: express your correct loop as a comprehension and run the same tests against both. Prefer the version you can explain clearly; do not expand the approved project scope.

### Save and continue

Keep your own code, independent expectations and observed evidence. Restart and rerun the completed practice cells to catch hidden state. Leave an unfinished optional problem clearly marked. Continue to the [scheduled project or assessment](#week-09-project-studio).

## Session 3 · Project studio


(week-09-project-studio)=
Apply this week's Python skills to the project component below. The supplied fixture, examples and helpers support your implementation; complete the marked student tasks and retain your own evidence.



### The people’s task this week

**Someone needs help:** Choose one small problem a volunteer actually faces.

**Your small part:** Implement your approved small change and check it with another learner.

![Week 9: the person’s need, your task, and the useful result.](assets/diagrams/course_week_09.svg)

**Picture one example:** Example: make pending pickups easier to find.

**A useful result:** A volunteer can try the changed feature and report what happened.

Use the supplied starting files and the task instructions below. The diagram explains the purpose; it adds no separate assignment.

In [ ]:
# @title Show this lesson’s embedded diagram(s)
try:
    from IPython.display import SVG, display
except ImportError:
    print("Diagrams are in assets/diagrams; Jupyter or Colab displays them below.")
else:
    display(SVG(filename='assets/diagrams/course_week_09.svg'))


### Project context

How can another volunteer verify that your change helps?

Campus Food Rescue is a fictional Cambodian university volunteer team working with nearby food businesses and community partner coordinators. All organizations, messages, quantities and approvals are invented. Counts are containers and declared planning portions, not money or a count of people served. A human coordinator approves suitability, partner arrangements and a collection window; Python records that decision and never certifies food safety.

**Project route.** Read the brief, inspect the supplied examples and complete each marked project task. Use a fresh kernel. Each week supplies its own fixture; earlier completed notebooks are not required. Student-work cells remain intentionally unfinished.

Use Python 3 with Jupyter. The programming examples use only Python's standard library. These activities run locally without an external service. Keep the three [fixture CSVs](https://loem-ms.github.io/programming-for-information-systems-site/learning_materials/final-data/offer_batches.csv) with these materials.

### The next need is a question a volunteer cannot yet answer

The coordinator can now distinguish a reservation from a collection and a delivery. Choose one unresolved need from Week 8. The change must preserve reviewed availability and lifecycle evidence, rather than turn an uncertain message into an automatic allocation.

**Let's think:** name the user, their next decision and the observation that would change after your feature. “Add AI” or “make a dashboard” is not yet an acceptance contract.

![A coordinator need becomes expected examples, implementation, review evidence and a revised change.](assets/diagrams/w09_change.svg)

In [ ]:
# @title Show this lesson’s embedded diagram(s)
try:
    from IPython.display import SVG, display
except ImportError:
    print("Diagrams are in assets/diagrams; Jupyter or Colab displays them below.")
else:
    display(SVG(filename='assets/diagrams/w09_change.svg'))


### Agree on acceptance

Use a local change request to agree on a scoped improvement and two expected outcomes with the classroom coordinator role. Freeze the examples before consulting optional code suggestions.

### Apply the method to your own approved change

Your assessed project addresses your Week 8 need; the supplied boundary repair is a demonstration. Preserve at least one unaffected behavior and explain how your test rejects a plausible wrong implementation. A small testable slice can be a read-only report, one validation field or a clearer handoff step. State inputs, outputs, boundary/failure behavior and what remains outside scope. A read-only feature should leave complete database snapshots unchanged. Missing events require follow-up rather than a claim that physical work failed.

### Implement the chosen change

Build your approved small Python or SQL change. Reuse supplied baseline records, and compare any optional coding suggestion with independent tests.

### 3. Establish the actual baseline

Load the authoritative CSVs and the complete supplied reservation/event engine. A new database is created each run. Headers, line snapshots and availability commit together; a failed later batch rolls back earlier changes. Events are whole-reservation observations and require collection before delivery. Do not add an event to a held or rejected request.

**Before running the next cells:** inspect the three fixture CSVs and write expected outcomes for all six pickup requests. Keep those predictions so you can compare them with the printed results.

In [ ]:
from pathlib import Path
import csv

def fixture_directory():
    for candidate in (Path("final-data"), Path("learning_materials/final-data"), Path("../learning_materials/final-data")):
        if (candidate / "offer_batches.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Keep learning_materials/final-data with the teaching materials")

DATA_DIR = fixture_directory()

def read_fixture():
    with (DATA_DIR / "offer_batches.csv").open(encoding="utf-8", newline="") as file:
        batches = [(row["batch_id"], row["description"], int(row["portions_per_container"]),
                    int(row["opening_available"])) for row in csv.DictReader(file)]
    grouped = {}
    with (DATA_DIR / "pickup_requests.csv").open(encoding="utf-8", newline="") as file:
        for row in csv.DictReader(file):
            if row["reviewed"] not in ("yes", "no"):
                raise ValueError("reviewed must be yes or no")
            reviewed = row["reviewed"] == "yes"
            request = grouped.setdefault(row["request_id"], {"reviewed": reviewed, "lines": []})
            if request["reviewed"] != reviewed:
                raise ValueError("conflicting review flags within one request")
            request["lines"].append((row["batch_id"], int(row["containers"])))
    requests = [(request_id, item["reviewed"], item["lines"])
                for request_id, item in grouped.items()]
    with (DATA_DIR / "rescue_events.csv").open(encoding="utf-8", newline="") as file:
        events = [(row["request_id"], row["event_type"]) for row in csv.DictReader(file)]
    return batches, requests, events

batches, requests, events = read_fixture()
print("Fixture:", len(batches), "batches;", len(requests), "requests;", len(events), "events")

### Supplied setup: run once, then continue

**No editing required in the engine below.** Run the fixture-loading cell above, then run this supplied engine cell to create the known starting database. Rerunning it resets the fixture; it does not retry an operation against the previous state.

The implementation may be collapsed by your notebook viewer. You may inspect it later, but your task is the behavior requested next. After setup, [continue to your project task](#week-09-project-task).

In [ ]:
import sqlite3

def pickup_label(total_portions, reviewed):
    if type(total_portions) is not int or total_portions < 0 or type(reviewed) is not bool:
        raise ValueError("expected nonnegative integer portions and a Boolean review flag")
    if not reviewed:
        return "hold"
    return "team" if total_portions >= 20 else "standard"

def new_rescue_db(batch_rows):
    db = sqlite3.connect(":memory:")
    db.execute("PRAGMA foreign_keys=ON")
    db.executescript("""
        CREATE TABLE batches(
            batch_id TEXT PRIMARY KEY, description TEXT,
            portions_per_container INTEGER CHECK(portions_per_container>0),
            opening_available INTEGER CHECK(opening_available>=0),
            available INTEGER CHECK(available>=0));
        CREATE TABLE reservations(
            request_id TEXT PRIMARY KEY, reviewed INTEGER CHECK(reviewed=1),
            label TEXT, planned_portions INTEGER CHECK(planned_portions>0));
        CREATE TABLE reservation_lines(
            request_id TEXT REFERENCES reservations(request_id), line_no INTEGER,
            batch_id TEXT REFERENCES batches(batch_id), containers INTEGER CHECK(containers>0),
            portions_per_container INTEGER CHECK(portions_per_container>0),
            PRIMARY KEY(request_id,line_no));
        CREATE TABLE events(
            request_id TEXT REFERENCES reservations(request_id),
            event_type TEXT CHECK(event_type IN ('collected','delivered')),
            PRIMARY KEY(request_id,event_type));
    """)
    with db:
        for batch_id, description, portions, available in batch_rows:
            if type(portions) is not int or portions <= 0 or type(available) is not int or available < 0:
                raise ValueError("invalid batch planning counts")
            db.execute("INSERT INTO batches VALUES (?,?,?,?,?)",
                       (batch_id, description, portions, available, available))
    return db

def reserve(db, request_id, reviewed, lines, inject_failure=False):
    if type(reviewed) is not bool or not reviewed:
        raise ValueError("coordinator review required")
    if not lines or any(type(count) is not int or count <= 0 for batch, count in lines):
        raise ValueError("reservation needs positive whole container counts")
    with db:
        prepared = []
        for batch_id, containers in lines:
            batch = db.execute("SELECT portions_per_container FROM batches WHERE batch_id=?",
                               (batch_id,)).fetchone()
            if batch is None:
                raise ValueError(f"unknown batch {batch_id}")
            prepared.append((batch_id, containers, batch[0]))
        total = sum(containers * portions for batch_id, containers, portions in prepared)
        label = pickup_label(total, reviewed)
        db.execute("INSERT INTO reservations VALUES (?,?,?,?)", (request_id, 1, label, total))
        for index, (batch_id, containers, portions) in enumerate(prepared, start=1):
            change = db.execute("UPDATE batches SET available=available-? WHERE batch_id=? AND available>=?",
                                (containers, batch_id, containers))
            if change.rowcount != 1:
                raise ValueError(f"insufficient available containers for {batch_id}")
            db.execute("INSERT INTO reservation_lines VALUES (?,?,?,?,?)",
                       (request_id, index, batch_id, containers, portions))
            if inject_failure and index == 1:
                raise RuntimeError("deliberate failure after first reservation line")
    return label

def record_event(db, request_id, event_type):
    if event_type not in ("collected", "delivered"):
        raise ValueError("unknown event type")
    with db:
        if db.execute("SELECT 1 FROM reservations WHERE request_id=?", (request_id,)).fetchone() is None:
            raise ValueError("event needs an accepted reservation")
        if event_type == "delivered" and db.execute(
            "SELECT 1 FROM events WHERE request_id=? AND event_type='collected'", (request_id,)
        ).fetchone() is None:
            raise ValueError("record collection before delivery")
        db.execute("INSERT INTO events VALUES (?,?)", (request_id, event_type))

def snapshot(db):
    # Table names are fixed teaching constants, never user-supplied SQL.
    return tuple(tuple(db.execute(f"SELECT * FROM {table} ORDER BY 1,2").fetchall())
                 for table in ("batches", "reservations", "reservation_lines", "events"))

def build_fixture(batch_rows, request_rows, event_rows):
    db = new_rescue_db(batch_rows)
    decisions = []
    for request_id, reviewed, lines in request_rows:
        before = snapshot(db)
        try:
            label = reserve(db, request_id, reviewed, lines)
        except ValueError as error:
            assert snapshot(db) == before
            decisions.append((request_id, "held" if not reviewed else "rejected", str(error)))
        else:
            decisions.append((request_id, "accepted", label))
    for request_id, event_type in event_rows:
        record_event(db, request_id, event_type)
    return db, decisions

# Rerunning this cell starts a new database; it does not retry against old state.
if "rescue_db" in globals():
    rescue_db.close()
rescue_db, decisions = build_fixture(batches, requests, events)
print("Reservation decisions:", decisions)


(week-09-project-task)=
### Your turn

Compare the observed outcomes with the six predictions you wrote before loading the baseline. Implement your own approved change in new cells or a separate module. Add report checks for offered/reserved/collected/delivered planning portions, batch totals and current available containers. Test unchanged state after unreviewed input, a later-line shortage and an injected failure. Test event ordering and duplicate-event behavior. For your feature, show a changed result, a preserved behavior, a read-only snapshot if applicable, and one limitation.

In [ ]:
# Write your implementation and evidence here. Add cells as needed.

### 4. Separate the rule, command and tests

A `.py` module can hold your function/query; an entry point obtains inputs and displays results; tests import the function without launching the application. Use `if __name__ == "__main__":` for startup. The supplied [pickup_app.py](https://loem-ms.github.io/programming-for-information-systems-site/learning_materials/programs/pickup_app.py) and [rescue_cli.py](https://loem-ms.github.io/programming-for-information-systems-site/learning_materials/programs/rescue_cli.py) demonstrate this with a planning calculation, not the full reservation system.

From `learning_materials/` in a terminal:

**Choose the course interpreter.** In terminal examples, `python` means the Python used for your course environment. If setup created `.venv`, replace it with the operating-system-specific path in the [program guide](https://loem-ms.github.io/programming-for-information-systems-site/content/learning-materials/programs/readme/#choose-the-course-interpreter). That path changes when you change directories. In a notebook, `import sys; print(sys.executable)` shows the interpreter in use.

```text
python programs/rescue_cli.py --containers 3
python -m unittest discover -s programs -p "test_*.py"
```

With the default three declared portions per container, three containers plan nine portions. That is not delivery evidence. The discovery command tests supplied practice modules, not your own project automatically. Provide the command and working directory that actually test your change. If using an optional project scaffold, replace TODOs and skipped checks before claiming completed behavior.

### Peer acceptance test

Exchange the local change request and run instructions. The reviewer chooses one additional distinguishing input; record the observation, decision, revision and retest.

### Your turn

Give a peer the starting fixture, actual run command/cells, two independent expected outcomes and one invariant. Record one observed finding and your response, including a retest. Individually explain one design judgment and one limitation. If working alone before class, label your self-review as rehearsal, not real peer evidence.

In [ ]:
# Write your implementation and evidence here. Add cells as needed.

### Handoff preparation for Week 10

**Offline reading card · A volunteer retries a job.** A volunteer loses the confirmation after reserving containers and repeats the request. A handoff note should identify the request ID, starting state, attempted action, observed result and recovery. Duplicate reservation IDs and duplicate event records need a deliberate response; neither should silently debit availability again.

There is no new research assignment. Use the supplied case to improve the project handoff: record the starting state, attempted action, observed result, recovery step and one check that would help a new coordinator trust the system.

**Handoff preparation time:** included in the Week 10 project estimate; this is not a seventh research assignment.

[Optional further reading (English)](https://sre.google/sre-book/postmortem-culture/)

### Optional supplied project references

These original examples explain the project application. Consult the reference you need; they are not extra Python core exercises.

### A function patch and independent test cases

The short example compares a strict comparison (`>`) with an inclusive one (`>=`). A Boolean gate returns before that comparison. The examples use project-flavored names to show general function control flow and regression checks.

In [ ]:
def before_label(total_portions, reviewed):
    if not reviewed:
        return "hold"
    return "team" if total_portions > 20 else "standard"  # Deliberate boundary mistake.

def after_label(total_portions, reviewed):
    if not reviewed:
        return "hold"
    return "team" if total_portions >= 20 else "standard"

for portions, reviewed, expected in [(19,True,"standard"),(20,True,"team"),(21,True,"team"),(20,False,"hold")]:
    print(portions,reviewed,before_label(portions,reviewed),after_label(portions,reviewed))
    assert after_label(portions,reviewed)==expected
assert before_label(20,True)!="team"
# Controlled inputs here are nonnegative integer planning counts and Boolean review flags.

Only the exact-boundary case changes when `>` becomes `>=`; the unreviewed case stays held. A small patch needs a stated rule and a case that distinguishes it from the old behavior. The general boundary practice is in Session 2 above.

### Select records with the intended predicate

A list comprehension can filter records using more than one Boolean field. The example distinguishes `not collected` from `not delivered`; these predicates select different records.

In [ ]:
def awaiting_collection(records):
    return [row["request_id"] for row in records if row["reserved"] and not row["collected"]]

records = [
 {"request_id":"D10","reserved":True,"collected":False,"delivered":False},
 {"request_id":"D11","reserved":True,"collected":True,"delivered":False},
 {"request_id":"D12","reserved":True,"collected":True,"delivered":True},
]
assert awaiting_collection(records)==["D10"]
# A plausible wrong query confuses collection follow-up with delivery follow-up.
wrong = [row["request_id"] for row in records if row["reserved"] and not row["delivered"]]
assert wrong==["D10","D11"]
print("Correct collection view:",awaiting_collection(records),"Wrong-stage view:",wrong)

D11 distinguishes the two filters: its collection field is True while its delivery field is False. Read each predicate explicitly before interpreting the returned list. The general Boolean-filter practice is in Session 2 above.


(week-09-finish)=
### Finish: check, save and decide what comes next

- [ ] I can explain the Python concept used this week.
- [ ] I kept a prediction, an observed result and an explanation for the core practice.
- [ ] I completed the required project behavior and checks described above.
- [ ] I recorded one peer observation or individual finding, plus a revision or defended decision.
- [ ] I stated one limitation or question that requires a person.

**Save:** save this notebook and any project files you changed. Keep a separate backup. Re-run the relevant completed cells from a known starting state before claiming the result; unresolved TODOs and skipped tests are not evidence of completion. Send assessed work to [submit.progmis@gmail.com](mailto:submit.progmis@gmail.com); exact deadlines and file names remain separately announced.

**Stop point:** when the required practice and project evidence are complete, save and stop. Optional follow-ups can wait.

**Next:** Week 10: demonstrate the improvement from clean data and hand it over.

[Back to Start Here](#week-09-start) · [Review the Python explanation](#week-09-session-1) · [Return to Project Studio](#week-09-project-studio)